# Fine-tuning Vision Transformers with vit-trainer

This notebook demonstrates how to use the `vit-trainer` package to fine-tune Vision Transformers for image classification.

**Author**: John Hodge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jman4162/PyTorch-Vision-Transformers-ViT/blob/main/notebooks/tutorial.ipynb)

## Installation

Install the vit-trainer package and its dependencies:

In [ ]:
# Install vit-trainer (uncomment if not installed)
# !pip install vit-trainer

## Quick Start

The simplest way to train a Vision Transformer:

In [ ]:
from vit_trainer import (
    Trainer,
    load_model,
    get_cifar10_loaders,
    CIFAR10_CLASSES,
)

# Load data
train_loader, val_loader, test_loader = get_cifar10_loaders(
    batch_size=64,
    seed=42,
)

print(f"Train: {len(train_loader.dataset)} samples")
print(f"Validation: {len(val_loader.dataset)} samples")
print(f"Test: {len(test_loader.dataset)} samples")

In [ ]:
# Load pretrained ViT model
model = load_model("vit_b_16", num_classes=10)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Create trainer with modern techniques
trainer = Trainer(
    model=model,
    lr=1e-4,
    weight_decay=0.05,
    warmup_epochs=2,
    use_amp=True,  # Mixed precision for faster training
)

# Train the model
history = trainer.fit(
    train_loader,
    val_loader,
    epochs=10,
    patience=3,
)

In [ ]:
# Evaluate on test set
loss, accuracy = trainer.evaluate(test_loader)
print(f"\nTest Accuracy: {accuracy:.2f}%")

## Visualizing Training Progress

In [ ]:
from vit_trainer.evaluation import plot_training_history

plot_training_history(history)

## Detailed Evaluation

In [ ]:
from vit_trainer import (
    get_predictions,
    compute_metrics,
    plot_confusion_matrix,
)

# Get all predictions
y_pred, y_true, probs = get_predictions(model, test_loader)

# Compute detailed metrics
metrics = compute_metrics(y_true, y_pred, CIFAR10_CLASSES)
print(metrics["classification_report"])

In [ ]:
# Plot confusion matrix
plot_confusion_matrix(y_true, y_pred, CIFAR10_CLASSES)

## Attention Visualization

One of the key benefits of Vision Transformers is interpretability through attention maps.

In [ ]:
from vit_trainer import visualize_samples_with_attention

# Visualize attention on test samples
visualize_samples_with_attention(
    model,
    test_loader.dataset,
    CIFAR10_CLASSES,
    num_samples=4,
)

## Single Image Prediction

In [ ]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
from vit_trainer import get_val_transform, visualize_attention, show_attention_on_image

def predict_single_image(model, image_path, device=None):
    """Predict class for a single image with attention visualization."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Load and transform image
    image = Image.open(image_path).convert("RGB")
    transform = get_val_transform()
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    model.to(device)
    
    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)[0]
        pred_idx = outputs.argmax(dim=1).item()
    
    # Get attention
    attn_map = visualize_attention(model, input_tensor[0], device=device)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    if attn_map is not None:
        axes[1].imshow(attn_map, cmap="hot")
        axes[1].set_title("Attention Map")
        axes[1].axis("off")
        
        overlay = show_attention_on_image(image.resize((224, 224)), attn_map)
        axes[2].imshow(overlay)
    else:
        axes[2].imshow(image)
    
    pred_label = CIFAR10_CLASSES[pred_idx]
    confidence = probs[pred_idx].item()
    axes[2].set_title(f"{pred_label}: {confidence:.1%}")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    # Print top-5 predictions
    print("\nTop-5 Predictions:")
    top_probs, top_indices = torch.topk(probs, 5)
    for prob, idx in zip(top_probs, top_indices):
        print(f"  {CIFAR10_CLASSES[idx]}: {prob.item():.2%}")

# Example usage (uncomment and provide your own image):
# predict_single_image(model, "path/to/your/image.jpg")

## Export to ONNX

In [ ]:
import torch.onnx

# Export to ONNX
model.eval()
model.cpu()

dummy_input = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy_input,
    "vit_cifar10.onnx",
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={
        "image": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
)

print("Model exported to vit_cifar10.onnx")

# Verify
import onnx
onnx_model = onnx.load("vit_cifar10.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model validation passed!")

## Using Configuration Files

In [ ]:
from vit_trainer import TrainingConfig

# Create a config
config = TrainingConfig(
    model_variant="vit_b_16",
    dataset="cifar10",
    batch_size=64,
    epochs=10,
    lr=1e-4,
    use_amp=True,
)

# Save config
config.save("my_config.yaml")
print("Config saved!")

# Load config
loaded_config = TrainingConfig.load("my_config.yaml")
print(f"Loaded config: {loaded_config.model_variant}, batch_size={loaded_config.batch_size}")

## CLI Usage

The package also provides a command-line interface:

```bash
# Train a model
vit-train train --model vit_b_16 --dataset cifar10 --epochs 10

# Evaluate a trained model
vit-train eval --checkpoint best_model.pt --dataset cifar10

# Predict on a single image
vit-train predict --checkpoint best_model.pt --image cat.jpg

# Export to ONNX
vit-train export --checkpoint best_model.pt --output model.onnx
```

## Conclusion

This tutorial demonstrated how to use the `vit-trainer` package for:

1. **Loading data**: `get_cifar10_loaders()` with proper train/val/test splits
2. **Loading models**: `load_model()` with pretrained weights
3. **Training**: `Trainer` class with AMP, warmup, early stopping
4. **Evaluation**: Metrics, confusion matrices, classification reports
5. **Visualization**: Attention maps for interpretability
6. **Export**: ONNX for deployment

For the original detailed tutorial, see `Fine_tuning_Vision_Transformers_ViT_with_PyTorch.ipynb`.